# Reranking

<a href="https://ecoop.github.io/rulebook/evaluation.html" target="_parent">Evaluation</a> ended on a miss that a score can't show you. Asked *How many betting rounds are there in
seven-card stud?*, retrieval puts Razz's betting-rounds rule first: the right topic, the wrong
game. This notebook explains why retrieval can't tell the difference, and adds the stage that can.

1. **What retrieval has been doing**: each text is embedded alone, then compared.
2. **Two stages**: retrieve a wider set cheaply, then reorder it more carefully.
3. **BM25 and hybrid**: reordering by shared words.
4. **A cross-encoder**: a model that reads the question and the passage together.
5. **Scores** for all of them on the poker and checkers gold sets.

This notebook runs **key-free**: every model is small and local, and nothing is billed. The
cross-encoder needs one extra install, `uv sync --extra rerank-local` (Colab gets it
automatically). It runs on Linux, Windows and Apple Silicon Macs but not on Intel Macs, where
the cross-encoder cells say so and the rest still runs.

On Colab, run the cells from the top; the first one fetches the repo (about a minute). On your
own machine, run `uv run jupyter lab` from the repo root.

In [ ]:
# On Colab, fetch the repo first (about a minute). On your own machine this is skipped.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/ecoop/rulebook /content/rulebook
    %pip install -q -e "/content/rulebook[rerank-local]"
    sys.path.insert(0, "/content/rulebook/src")  # this session can't see the new install otherwise

from rulebook.notebook import hit_cell, setup, show_table
setup()

## 1. What retrieval has been doing

Every retrieval in the other notebooks works in three steps:

1. **When the index is built**, each passage's vector is computed *on its own*. The model
   reads Razz's rule and writes down 256 numbers. It has never seen a question.
2. **When a question comes in**, the question's vector is computed *on its own*, without
   seeing any passage.
3. **Comparing** them is arithmetic between the two vectors (their distance). No model runs at
   this step.

```
passage  ──► embed ──► vector ─┐
                               ├──► distance
question ──► embed ──► vector ─┘
```

The only place the question and the passage meet is that distance, after each has been
squeezed into a fixed-size summary separately. A passage's vector has to suit every question
it might ever answer, so in practice it records what the passage is *about*. Here is the
question from Evaluation with its five nearest passages, in the default model's index of the
poker rules:

In [ ]:
from rulebook.config import settings
from rulebook.eval import build_scratch_index, load_gold

index = build_scratch_index("poker", settings.embedding_model)
gold = load_gold("poker")

VARIANT = {"1": "five-card draw", "2": "five-card stud", "3": "seven-card stud", "4": "Razz", "5": "Omaha"}


def named(rule_id):
    """'4.4' -> '4.4 (Razz)'."""
    variant = VARIANT.get(rule_id.split(".")[0])
    return f"{rule_id} ({variant})" if variant else rule_id


def candidates(question, n=20):
    """The n nearest passages by embedding distance: what retrieval returns."""
    [q_vec] = index.embedder.embed([question], input_type="query")
    return index.store.search(q_vec, k=n)


QUESTION = "How many betting rounds are there in seven-card stud?"
show_table(
    ["rank", "rule", "distance", "passage"],
    [[n, named(r["rule_id"]), f"{r['_distance']:.3f}", r["text"][:110]]
     for n, r in enumerate(candidates(QUESTION, 5), start=1)],
)

Rule 3.4 answers the question. Rule 4.4 is about Razz, but it has the same words, *five betting
rounds* and *seven-card stud*, so its vector lands nearest. To rule it out, a model has to
relate the two texts: the passage's subject is Razz, and the question's subject is seven-card
stud. A vector made before the question existed can't hold that, and a distance can only
measure overlap.

## 2. Two stages: widen, then rerank

A model that reads the question and the passage *together* can relate them. The catch is that
nothing can be computed in advance, because the result depends on the question: it has to run
once per question for every passage it scores. Fine for 20 passages, impossible for millions.

So reranking adds a second stage:

- **Stage 1 (retrieval, cheap):** take the nearest 20 passages instead of 5. Its job is just to
  get the right passage *into* the 20.
- **Stage 2 (the reranker, slower):** score those 20 and keep the best 5. Its job is the
  *order*.

Stage 1 is already doing its job. For every poker question that retrieval gets wrong at rank 1,
here is where the cited rule actually sits:

In [ ]:
def rank_in(rule_ids, rows):
    """1-indexed rank of the first cited rule in rows, or None."""
    return next((n for n, r in enumerate(rows, start=1) if r["rule_id"] in rule_ids), None)


rows = []
for item in gold:
    pool = candidates(item.question)
    if pool[0]["rule_id"] not in item.rule_ids:
        rows.append([item.question, " / ".join(map(named, item.rule_ids)),
                     named(pool[0]["rule_id"]), rank_in(item.rule_ids, pool)])
show_table(["question", "cited", "came first", "cited rule's rank (of 20)"], rows)

Every cited rule is in the top 20, most of them right behind the passage that came first. The
answer is in the pool; the order is what's wrong, and the order is what a reranker changes.

## 3. Reranking by shared words: BM25 and hybrid

The oldest reranker needs no model at all. **BM25** scores a passage by the question's words it
contains:

- a word that few passages use (*Omaha*, *bring-in*) counts for more than a common one
  (*cards*);
- a word repeated in a passage helps, but each repeat helps less;
- long passages are discounted a little, so they don't win just by containing more words.

It's good at exact terms that embeddings can blur, but it still only matches words; it doesn't
understand them. **Hybrid** combines the two orders: each passage gets a score from its rank in
the embedding order and its rank in the BM25 order (*reciprocal rank fusion*), so a passage has
to do well on both to rise. Both are in `src/rulebook/rerank.py`, about 60 lines.

Here is what each does with the Razz question:

In [ ]:
from rulebook.rerank import BM25Reranker, HybridReranker, reorder


def top(question, reranker, n=5):
    """Rerank the 20 candidates; return the top n rule ids, named."""
    pool = candidates(question)
    if reranker is not None:
        pool = reorder(question, pool, [r["text"] for r in pool], reranker)
    return ", ".join(named(r["rule_id"]) for r in pool[:n])


bm25, hybrid = BM25Reranker(), HybridReranker()
show_table(
    ["order", "top 3"],
    [["embedding only", top(QUESTION, None, 3)],
     ["BM25", top(QUESTION, bm25, 3)],
     ["hybrid", top(QUESTION, hybrid, 3)]],
)

Word overlap can't separate them either: Razz's rule contains *seven-card stud* and *betting
rounds* word for word.

## 4. A cross-encoder: reading the question and the passage together

A **cross-encoder** makes one input out of both texts and runs a transformer over it:

```
[question] [SEP] [passage]  ──► transformer ──► one score
```

Inside the transformer, every word of the question can attend to every word of the passage,
layer after layer: *seven-card stud* in the question can look at *In Razz* and at *exactly as in
seven-card stud* in the passage. The model was trained on search queries paired with the
passages that answer them, to output one number: does this passage answer this question?
That is the difference between "about it" and "answers it".

The one here is `ms-marco-MiniLM-L-6-v2` (22M parameters, about 80 MB, downloaded once), run
on [ONNX](https://onnxruntime.ai/) through [fastembed](https://github.com/qdrant/fastembed). It
is small; a bigger cross-encoder, or an LLM given the question and the 20 passages, does the
same job better and slower.

In [ ]:
from rulebook.rerank import CrossEncoderReranker

try:
    cross = CrossEncoderReranker()
except ImportError as e:
    cross = None
    print(f"Skipping the cross-encoder here: {e}")

if cross is not None:
    pool = candidates(QUESTION)
    scores = cross.rerank(QUESTION, [r["text"] for r in pool])
    ranked = sorted(zip(scores, pool, strict=True), key=lambda pair: -pair[0])[:5]
    show_table(
        ["rank", "rule", "cross-encoder score", "embedding rank"],
        [[n, named(r["rule_id"]), f"{s:.2f}", pool.index(r) + 1]
         for n, (s, r) in enumerate(ranked, start=1)],
    )

The score is the model's raw output (higher = a better answer); only the order matters, not the
scale. The last column is where embedding distance had each passage.

## 5. Scores

One question shows the idea; the gold sets show whether it holds up. The same golds and the same
scores as in Evaluation (hit@1, hit@3, MRR), with each reranker reordering the top 20, plus the
time each setup takes per question. As there, a hit@k cell shows the count too: **0.64 (14/22)**
means 14 of the 22 poker questions had a cited rule in the top *k*.

In [ ]:
import time

from rulebook.eval import evaluate

checkers_index = build_scratch_index("checkers", settings.embedding_model)
SETS = {"poker": (gold, index), "checkers": (load_gold("checkers"), checkers_index)}
RERANKERS = {"embedding only": None, "BM25": bm25, "hybrid": hybrid}
if cross is not None:
    RERANKERS["cross-encoder"] = cross

results = {}
rows = []
for rule_set, (golds, idx) in SETS.items():
    for name, reranker in RERANKERS.items():
        start = time.perf_counter()
        res = evaluate(golds, k=5, index=idx, reranker=reranker, candidates=20)
        ms = 1000 * (time.perf_counter() - start) / len(golds)
        results[rule_set, name] = res
        rows.append([rule_set, name, hit_cell(res, 1), hit_cell(res, 3), f"{res.mrr:.2f}", f"{ms:.0f}"])
show_table(["rule set", "reranker", "hit@1", "hit@3", "MRR", "ms / question"], rows)

A total can hide trades: a reranker can fix some questions and break others. Here is every
question whose first passage changed from right to wrong, or wrong to right:

In [ ]:
rows = []
for rule_set in SETS:
    base = results[rule_set, "embedding only"]
    for name in RERANKERS:
        if name == "embedding only":
            continue
        for b, r in zip(base.items, results[rule_set, name].items, strict=True):
            was, now = b.retrieved_ids[0] in b.item.rule_ids, r.retrieved_ids[0] in r.item.rule_ids
            if was != now:
                first = named(r.retrieved_ids[0]) if rule_set == "poker" else r.retrieved_ids[0]
                rows.append([rule_set, name, "fixed" if now else "broke", r.item.question, first])
show_table(["rule set", "reranker", "", "question", "now first"], rows)

The cross-encoder moves poker's hit@1 from 0.64 (14/22) to 0.95 (21/22) and breaks
nothing. Of the 8 questions retrieval got wrong at rank 1, it puts the cited rule first for 7, the seven-card stud betting
question included; only *In Razz, how is the hand dealt?* still misses. On checkers it fixes
three and breaks none (0.50 to 0.75, 6/12 to 9/12). It doesn't fix everything: *How do you win a game of
checkers?*, the question from Embeddings, still gets the wrong passage first.

BM25 on its own is *worse* than no reranker at hit@1, on both rule sets: it fixes some questions
but breaks more. One of its breaks is the wrong-variant mistake itself: *In Razz, does the
high card or the low card post the bring-in?* gets five-card stud's bring-in rule, which shares
the question's words. Hybrid keeps both of BM25's poker fixes and none of its four poker breaks
(one checkers break remains), which is the reason to fuse the two orders rather than replace one with the other.

The price of the cross-encoder is time: a few hundred milliseconds per question to score 20
candidates on the machine that builds this page (it varies from run to run), against about 1 ms
for the others. Next to the seconds Claude
takes to write an answer that's small, and it's why the cross-encoder reranks 20 passages
rather than the whole index.

## 6. Turning it on in the app

Reranking is a setting, off by default. `RERANK_PROVIDER` picks the reranker (`none`, `bm25`,
`hybrid`, or `local` for the cross-encoder) and `RERANK_CANDIDATES` sets how many passages
stage 1 hands it. In `.env`:

```
RERANK_PROVIDER=local
RERANK_CANDIDATES=20
```

(On an Intel Mac, where the cross-encoder can't install, `hybrid` is the next best here.)

`retrieve()` then takes the nearest `RERANK_CANDIDATES` passages and keeps the reranker's top
`k`. The eval command line takes the same choice, so you can score your own rule sets before
switching it on:

```bash
uv run python -m rulebook.eval --domain poker --rerank local
```

## Takeaway

- Retrieval embeds the question and each passage **separately**, so it finds passages *about*
  the question. On poker, that includes the same rule from the wrong variant.
- A **reranker** takes a second, slower look at a wider set of candidates. **BM25** and
  **hybrid** reorder by shared words; a **cross-encoder** reads the question and the passage
  together, the only one of the three that can tell "about it" from "answers it". On poker it
  puts the cited rule first for 21 of 22 questions, up from 14.
- Retrieval's job becomes getting the right passage into the candidates; the reranker's job is
  the order.
- Score it before you turn it on, on the gold set: a reranker can fix some questions and break
  others.

**Next: <a href="https://ecoop.github.io/rulebook/chunking.html" target="_parent">Chunking</a>** (<a href="https://colab.research.google.com/github/ecoop/rulebook/blob/main/notebooks/chunking.ipynb" target="_parent">run it in Colab</a>). A cheaper fix for the wrong-variant misses, one
step earlier: when the rules are cut into chunks. Issue
<a href="https://github.com/ecoop/rulebook/issues/258" target="_parent">#258</a> tracks hosted rerankers (Voyage, Cohere) and Claude as a reranker.